# Thumbnail Visualization Pipeline — 5개 카테고리

`YT_dataset_*.csv` 다섯 종류(health, food, VLOG, society, education)에 대해  
**DINOv2-base**와 **SigLIP2-base**로 썸네일 임베딩을 추출하고, **단색** UMAP / t-SNE로 분포만 확인합니다.

## 데이터셋

| 카테고리 | 입력 CSV |
|---|---|
| health | `YT_dataset_health.csv` |
| food | `YT_dataset_food.csv` |
| VLOG | `YT_dataset_VLOG.csv` |
| society | `YT_dataset_society.csv` |
| education | `YT_dataset_education.csv` |

## 출력 구조 (카테고리별)

```
artifacts/{카테고리}/
  dataset_ready.csv
  prepare_summary.json
  embeddings/dinov2-base/   embeddings.npy, metadata.csv, run_info.json
  embeddings/siglip2-base/  (동일)
  visualizations/dinov2-base/  umap.png, tsne_perp*.png
  visualizations/siglip2-base/ (동일)
```

## 파이프라인 단계

```
Step 1: prepare   → 썸네일 경로 검증, dataset_ready.csv (카테고리별)
Step 2: extract   → dinov2-base + siglip2-base 임베딩 (카테고리별)
Step 3: visualize → 단색 UMAP / t-SNE (카테고리·모델별)
```


---
## 라이브러리 & 전역 설정


In [ ]:
from __future__ import annotations

import csv
import json
import os
import re
import sys
from collections import Counter
from dataclasses import dataclass
from pathlib import Path
from typing import Dict, Iterable, List, Optional, Sequence, Tuple

# ── 작업 디렉터리 (프로젝트 루트) ──
_marker = Path("YT_dataset_health.csv")
if not _marker.exists():
    for p in Path.cwd().rglob("YT_dataset_health.csv"):
        os.chdir(p.parent)
        break
print(f"CWD: {Path.cwd()}")

# ── 5개 데이터셋 (카테고리 키, CSV 경로) ──
DATASETS: List[Tuple[str, Path]] = [
    ("health",    Path("YT_dataset_health.csv")),
    ("food",      Path("YT_dataset_food.csv")),
    ("VLOG",      Path("YT_dataset_VLOG.csv")),
    ("society",   Path("YT_dataset_society.csv")),
    ("education", Path("YT_dataset_education.csv")),
]

ARTIFACTS_BASE = Path("artifacts")

# ── 모델 ──
MODEL_ALIASES = {
    "dinov2-base":  "facebook/dinov2-base",
    "siglip2-base": "google/siglip2-base-patch16-224",
}
MODELS_TO_RUN = "dinov2-base,siglip2-base"
BATCH_SIZE    = 32
DEVICE        = "auto"   # "auto" | "cpu" | "cuda"

UMAP_NEIGHBORS    = 20
UMAP_MIN_DIST     = 0.1
TSNE_PERPLEXITIES = [5, 15, 30]

print("설정 완료")
print(f"  카테고리: {[c for c, _ in DATASETS]}")
print(f"  모델: {MODELS_TO_RUN}")
print(f"  산출 루트: {ARTIFACTS_BASE}/{{카테고리}}/")


---
## 공통 유틸리티 함수


In [ ]:
def ensure_dir(path: Path) -> None:
    path.mkdir(parents=True, exist_ok=True)


def read_csv_rows(path: Path, encoding: str = "utf-8-sig") -> List[Dict[str, str]]:
    with path.open("r", encoding=encoding, newline="") as f:
        return list(csv.DictReader(f))


def write_csv_rows(
    path: Path,
    fieldnames: Sequence[str],
    rows: Iterable[Dict],
) -> None:
    with path.open("w", encoding="utf-8", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        for row in rows:
            writer.writerow(row)


print("유틸리티 함수 정의 완료")


---
## Step 1: 데이터 준비 (`prepare`)

각 카테고리 CSV에서 **썸네일 파일이 존재하는 행만** `artifacts/{카테고리}/dataset_ready.csv`로 저장합니다.


In [ ]:
def prepare_dataset(youtube_csv: Path, output_dir: Path) -> None:
    ensure_dir(output_dir)
    rows = read_csv_rows(youtube_csv)

    ready, skipped_no_path, skipped_missing = [], 0, 0
    for row in rows:
        thumb_str = (row.get("thumbnail_path") or "").strip()
        if not thumb_str:
            skipped_no_path += 1
            continue
        thumb = Path(thumb_str)
        if not thumb.exists():
            skipped_missing += 1
            continue
        ready.append({
            "channel_name":   (row.get("channel_name")  or "").strip(),
            "channel_id":     (row.get("channel_id")    or "").strip(),
            "video_id":       (row.get("video_id")      or "").strip(),
            "title":          (row.get("title")         or "").strip(),
            "published_at":   (row.get("published_at")  or "").strip(),
            "duration":       (row.get("duration")      or "").strip(),
            "thumbnail_path": thumb_str,
            "thumbnail_url":  (row.get("thumbnail_url") or "").strip(),
        })

    fieldnames = [
        "channel_name", "channel_id", "video_id",
        "title", "published_at", "duration",
        "thumbnail_path", "thumbnail_url",
    ]
    write_csv_rows(output_dir / "dataset_ready.csv", fieldnames, ready)

    summary = {
        "source_csv":        str(youtube_csv),
        "total_input":       len(rows),
        "skipped_no_path":   skipped_no_path,
        "skipped_missing":   skipped_missing,
        "ready_count":       len(ready),
    }
    with (output_dir / "prepare_summary.json").open("w", encoding="utf-8") as f:
        json.dump(summary, f, ensure_ascii=False, indent=2)

    print(f"[{output_dir.name}] 입력 {len(rows)} → 준비 {len(ready)} (경로없음 {skipped_no_path}, 파일없음 {skipped_missing})")


print("prepare 함수 정의 완료")


In [ ]:
# ── Step 1 실행 (5개 카테고리) ──
for cat, csv_path in DATASETS:
    if not csv_path.exists():
        print(f"[{cat}] 건너뜀 — 파일 없음: {csv_path}")
        continue
    out = ARTIFACTS_BASE / cat
    prepare_dataset(youtube_csv=csv_path, output_dir=out)


---
## Step 2: 임베딩 추출 (`extract`)

카테고리별 `dataset_ready.csv`에 대해 **dinov2-base**, **siglip2-base** 순으로 임베딩을 저장합니다.

> GPU 사용: `DEVICE = "cuda"`


In [ ]:
import numpy as np
from PIL import Image


@dataclass
class LoadedModel:
    model_id:  str
    alias:     str
    processor: object
    model:     object
    device:    str


def _resolve_device(requested: str) -> str:
    if requested != "auto":
        return requested
    try:
        import torch
        if torch.cuda.is_available():
            return "cuda"
    except Exception:
        pass
    return "cpu"


def _resolve_model_id(alias_or_id: str) -> Tuple[str, str]:
    alias_or_id = alias_or_id.strip()
    if alias_or_id in MODEL_ALIASES:
        return alias_or_id, MODEL_ALIASES[alias_or_id]
    safe_alias = re.sub(r"[^a-zA-Z0-9._-]+", "_", alias_or_id)
    return safe_alias, alias_or_id


def _load_transformers_model(alias_or_id: str, device: str) -> LoadedModel:
    from transformers import AutoImageProcessor, AutoModel, AutoProcessor
    alias, model_id = _resolve_model_id(alias_or_id)
    model = AutoModel.from_pretrained(model_id)
    try:
        processor = AutoProcessor.from_pretrained(model_id)
    except Exception:
        processor = AutoImageProcessor.from_pretrained(model_id)
    model = model.to(device).eval()
    return LoadedModel(model_id=model_id, alias=alias,
                       processor=processor, model=model, device=device)


def _extract_batch_embeddings(loaded: LoadedModel, images: list) -> list:
    import torch
    inputs = loaded.processor(images=images, return_tensors="pt")
    inputs = {k: v.to(loaded.device) for k, v in inputs.items()
              if isinstance(v, torch.Tensor)}
    with torch.no_grad():
        outputs = loaded.model(**inputs)
    if hasattr(outputs, "pooler_output") and outputs.pooler_output is not None:
        feats = outputs.pooler_output
    else:
        feats = outputs.last_hidden_state[:, 0, :]
    feats = feats / feats.norm(dim=-1, keepdim=True)
    return feats.cpu().float().numpy().tolist()


def extract_embeddings(
    dataset_csv: Path,
    output_root: Path,
    models: str = MODELS_TO_RUN,
    batch_size: int = BATCH_SIZE,
    device: str = DEVICE,
) -> None:
    ensure_dir(output_root)
    rows = read_csv_rows(dataset_csv, encoding="utf-8")

    if not rows:
        raise RuntimeError(f"데이터가 없습니다: {dataset_csv}")

    device_str  = _resolve_device(device)
    model_specs = [m.strip() for m in models.split(",") if m.strip()]
    print(f"추출 대상: {len(rows)}행 | 디바이스: {device_str} | 모델: {model_specs}")

    for model_name in model_specs:
        print(f"\n[{model_name}] 모델 로드 중...")
        loaded    = _load_transformers_model(model_name, device=device_str)
        model_dir = output_root / loaded.alias
        ensure_dir(model_dir)

        all_embs, all_meta = [], []
        batch_imgs, batch_meta_buf = [], []

        def flush():
            if not batch_imgs:
                return
            for i, emb in enumerate(_extract_batch_embeddings(loaded, batch_imgs)):
                all_embs.append(emb)
                all_meta.append(batch_meta_buf[i])
            batch_imgs.clear()
            batch_meta_buf.clear()

        missing_thumb = 0
        for row in rows:
            thumb = Path(row["thumbnail_path"])
            if not thumb.exists():
                missing_thumb += 1
                continue
            try:
                img = Image.open(thumb).convert("RGB")
            except Exception:
                continue
            batch_imgs.append(img)
            batch_meta_buf.append({
                "video_id":       row.get("video_id", ""),
                "channel_id":     row.get("channel_id", ""),
                "channel_name":   row.get("channel_name", ""),
                "published_at":   row.get("published_at", ""),
                "duration":       row.get("duration", ""),
                "thumbnail_path": row.get("thumbnail_path", ""),
            })
            if len(batch_imgs) >= batch_size:
                flush()
        flush()

        if missing_thumb:
            print(f"  썸네일 파일 없음(건너뜀): {missing_thumb}개")
        if not all_embs:
            print(f"  [{model_name}] 임베딩 없음, 건너뜀")
            continue

        embeddings = np.stack(all_embs, axis=0)
        np.save(model_dir / "embeddings.npy", embeddings)

        meta_fields = ["video_id", "channel_id", "channel_name",
                       "published_at", "duration", "thumbnail_path"]
        write_csv_rows(model_dir / "metadata.csv", meta_fields, all_meta)

        run_info = {
            "alias": loaded.alias, "model_id": loaded.model_id, "device": device_str,
            "batch_size": batch_size, "rows_input": len(rows),
            "rows_embedded": int(embeddings.shape[0]),
            "embedding_dim": int(embeddings.shape[1]),
        }
        with (model_dir / "run_info.json").open("w", encoding="utf-8") as f:
            json.dump(run_info, f, ensure_ascii=False, indent=2)

        print(f"  임베딩 shape: {embeddings.shape} → {model_dir / 'embeddings.npy'}")


print("extract 함수 정의 완료")


In [ ]:
# ── Step 2 실행 (5개 카테고리 × 2모델) ──
for cat, _ in DATASETS:
    ready = ARTIFACTS_BASE / cat / "dataset_ready.csv"
    if not ready.exists():
        print(f"[{cat}] dataset_ready.csv 없음, 건너뜀")
        continue
    emb_root = ARTIFACTS_BASE / cat / "embeddings"
    print(f"\n========== [{cat}] 임베딩 추출 ==========")
    extract_embeddings(
        dataset_csv=ready,
        output_root=emb_root,
        models=MODELS_TO_RUN,
        batch_size=BATCH_SIZE,
        device=DEVICE,
    )


---
## Step 3: 시각화 (`visualize`)

**단색** scatter로 UMAP / t-SNE만 저장합니다 (연령·레이블 색 구분 없음).


In [ ]:
def _scatter(path: Path, coords, title: str, color: str = "#4C72B0") -> None:
    """단색 2D scatter plot 저장."""
    import matplotlib.pyplot as plt
    fig, ax = plt.subplots(figsize=(9, 7))
    ax.scatter(coords[:, 0], coords[:, 1], s=10, alpha=0.45, color=color, linewidths=0)
    ax.set_title(title, fontsize=13)
    ax.set_xlabel("dim-1")
    ax.set_ylabel("dim-2")
    fig.tight_layout()
    fig.savefig(path, dpi=180)
    plt.close(fig)
    print(f"  저장: {path}")


def visualize_category_models(
    category: str,
    embeddings_root: Path,
    output_dir: Path,
    umap_neighbors: int = UMAP_NEIGHBORS,
    umap_min_dist: float = UMAP_MIN_DIST,
    tsne_perplexities: List[int] = TSNE_PERPLEXITIES,
) -> None:
    import numpy as np
    import umap as umap_lib
    from sklearn.manifold import TSNE

    model_dirs = sorted([p for p in embeddings_root.iterdir() if p.is_dir()])
    if not model_dirs:
        print(f"[{category}] 임베딩 폴더 없음: {embeddings_root}")
        return

    ensure_dir(output_dir)

    for model_dir in model_dirs:
        emb_path = model_dir / "embeddings.npy"
        if not emb_path.exists():
            print(f"[{category}/{model_dir.name}] embeddings.npy 없음, 건너뜀")
            continue

        X = np.load(emb_path)
        print(f"\n[{category} / {model_dir.name}] shape: {X.shape}")

        model_out = output_dir / model_dir.name
        ensure_dir(model_out)

        reducer = umap_lib.UMAP(
            n_components=2, n_neighbors=umap_neighbors,
            min_dist=umap_min_dist, metric="cosine", random_state=42,
        )
        umap_coords = reducer.fit_transform(X)
        _scatter(
            model_out / "umap.png",
            umap_coords,
            title=f"UMAP — {category} / {model_dir.name} (n={X.shape[0]})",
        )

        for p in tsne_perplexities:
            if p >= X.shape[0]:
                print(f"  t-SNE perp={p} 건너뜀 (샘플 부족)")
                continue
            coords = TSNE(
                n_components=2, perplexity=p, init="pca",
                learning_rate="auto", random_state=42,
            ).fit_transform(X)
            _scatter(
                model_out / f"tsne_perp{p}.png",
                coords,
                title=f"t-SNE perp={p} — {category} / {model_dir.name} (n={X.shape[0]})",
            )

        summary = {
            "category": category,
            "model": model_dir.name,
            "n_samples": int(X.shape[0]),
            "embedding_dim": int(X.shape[1]),
        }
        with (model_out / "visualize_summary.json").open("w", encoding="utf-8") as f:
            json.dump(summary, f, ensure_ascii=False, indent=2)

    print(f"[{category}] 시각화 완료 → {output_dir}")


print("visualize 함수 정의 완료")


In [ ]:
# ── Step 3 실행 (5개 카테고리) ──
for cat, _ in DATASETS:
    emb_root = ARTIFACTS_BASE / cat / "embeddings"
    viz_root = ARTIFACTS_BASE / cat / "visualizations"
    visualize_category_models(
        category=cat,
        embeddings_root=emb_root,
        output_dir=viz_root,
        umap_neighbors=UMAP_NEIGHBORS,
        umap_min_dist=UMAP_MIN_DIST,
        tsne_perplexities=TSNE_PERPLEXITIES,
    )


---
## 결과 인라인 확인

카테고리 → 모델별 UMAP / t-SNE PNG를 노트북에 표시합니다.


In [ ]:
import matplotlib.pyplot as plt
import matplotlib.image as mpimg


def show_images(paths, cols: int = 2, title_prefix: str = "") -> None:
    paths = [Path(p) for p in paths if Path(p).exists()]
    if not paths:
        print("표시할 이미지가 없습니다.")
        return
    rows = (len(paths) + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=(cols * 7, rows * 5.5))
    axes = [axes] if rows * cols == 1 else list(
        axes.flat if hasattr(axes, "flat") else [axes]
    )
    for ax, p in zip(axes, paths):
        ax.imshow(mpimg.imread(p))
        ax.set_title(f"{title_prefix}{p.stem}", fontsize=10)
        ax.axis("off")
    for ax in axes[len(paths):]:
        ax.axis("off")
    plt.tight_layout()
    plt.show()


for cat, _ in DATASETS:
    viz = ARTIFACTS_BASE / cat / "visualizations"
    if not viz.is_dir():
        continue
    for model_dir in sorted(viz.iterdir()):
        if not model_dir.is_dir():
            continue
        imgs = sorted(model_dir.glob("*.png"))
        if imgs:
            print(f"\n=== {cat} / {model_dir.name} ===")
            show_images(imgs, cols=2)
